# 🫁 Pneumonia Detection - V2
### Dataset: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
### ZIP: /content/drive/MyDrive/Datasets/chest-xray-pneumonia.zip
### Model: MobileNetV2 with preprocess_input

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

zip_path = '/content/drive/MyDrive/Datasets/chest-xray-pneumonia.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/pneumonia')
print('Extracted!')

train_dir = test_dir = None
for root, dirs, files in os.walk('/content/pneumonia'):
    dl = [d.upper() for d in dirs]
    if 'NORMAL' in dl and 'PNEUMONIA' in dl:
        if 'train' in root.lower(): train_dir = root
        elif 'test' in root.lower() or 'val' in root.lower(): test_dir = root

print('Train:', train_dir)
print('Test: ', test_dir)
print('Classes:', os.listdir(train_dir))
for cls in os.listdir(train_dir):
    print(f'  {cls}: {len(os.listdir(os.path.join(train_dir, cls)))} images')

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32

# preprocess_input — MobileNetV2 ka sahi preprocessing [-1, 1]
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_gen.flow_from_directory(
    train_dir, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=True
)
test_data = test_gen.flow_from_directory(
    test_dir, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)

# Expected: {'NORMAL': 0, 'PNEUMONIA': 1}
print('Classes:', train_data.class_indices)
print('Train:', train_data.samples, '| Test:', test_data.samples)

# Class weights
labels  = train_data.classes
weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weights = {i: w for i, w in enumerate(weights)}
print('Class weights:', class_weights)

In [ ]:
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x   = base(inp, training=False)
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.Dropout(0.5)(x)
x   = layers.Dense(128, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation='sigmoid')(x)

model = Model(inp, out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Phase 1: Train top layers
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.3, patience=2, monitor='val_loss', min_lr=1e-6)
]

history1 = model.fit(
    train_data, validation_data=test_data,
    epochs=10, callbacks=callbacks,
    class_weight=class_weights
)
print(f'Phase 1 Best: {max(history1.history["val_accuracy"])*100:.2f}%')

In [ ]:
# Phase 2: Fine-tune last 30 layers
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks2 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.3, patience=2, monitor='val_loss', min_lr=1e-7)
]

history2 = model.fit(
    train_data, validation_data=test_data,
    epochs=10, callbacks=callbacks2,
    class_weight=class_weights
)
print(f'Phase 2 Best: {max(history2.history["val_accuracy"])*100:.2f}%')

In [ ]:
loss, acc = model.evaluate(test_data)
print(f'\nTest Accuracy: {acc*100:.2f}%')

preds        = (model.predict(test_data) > 0.5).astype(int).flatten()
true_classes = test_data.classes
class_names  = list(test_data.class_indices.keys())

print('\nConfusion Matrix:')
print(confusion_matrix(true_classes, preds))
print('\nClassification Report:')
print(classification_report(true_classes, preds, target_names=class_names))

In [ ]:
save_dir = '/content/drive/MyDrive/ml_models'
os.makedirs(save_dir, exist_ok=True)
model.save(f'{save_dir}/pneumonia_model.h5')
print('✅ pneumonia_model.h5 saved!')
print('Classes:', train_data.class_indices)